In [5]:
# Import library require for process
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
import warnings as ws
ws.filterwarnings('ignore')
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis 



def lda_selection(indep_X, dep_Y, n_components):
    sc = StandardScaler()
    X_scaled = sc.fit_transform(indep_X)
    
    # LDA constraint: n_components can be at most (number of classes - 1)
    max_components = dep_Y.nunique() - 1
    if n_components > max_components:
        print(f"Warning: n_components reduced from {n_components} to {max_components} (max allowed = classes - 1)")
        n_components = max_components
    
    lda = LinearDiscriminantAnalysis(n_components=n_components)
    X_lda = lda.fit_transform(X_scaled, dep_Y)   # LDA needs dep_Y, unlike PCA
    
    print("Explained variance ratio:", lda.explained_variance_ratio_)
    print("Total variance captured:", sum(lda.explained_variance_ratio_))
    
    lda_df = pd.DataFrame(
        X_lda,
        columns=[f'LD{i+1}' for i in range(n_components)],
        index=indep_X.index
    )
    return lda_df

# split_scalar - Split the input, output train and test set. then changes the input to scalar value
# Note: PCA output is already scaled (since we scaled before PCA), so scaling again here
# is redundant but harmless — kept as-is to match your original pipeline structure.
def split_scalar(indep_X, dep_Y):
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size=0.25, random_state=0)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)
    return X_train, X_test, y_train, y_test


# cm_prediction - used for classification method, model prediction evaluate confusion matrix and send accuracy, Report and send back
# FIX: y_test is now passed in explicitly instead of relying on a global variable
def cm_prediction(classifier, X_test, y_test):
    y_pred = classifier.predict(X_test)

    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, y_pred)

    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report

    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return classifier, Accuracy, report, X_test, y_test, cm


# logistic method is used for logistic regression model creation and evaluate confusion matrix and send accuracy, Report
def logistic(X_train, y_train, X_test, y_test):
    from sklearn.linear_model import LogisticRegression
    classifier = LogisticRegression(random_state=0, max_iter=1000)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


# svm_linear method is used for svm_linear model creation and evaluate confusion matrix and send accuracy, Report
def svm_linear(X_train, y_train, X_test, y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel='linear', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


# svm_NL method is used for svm_NL model creation and evaluate confusion matrix and send accuracy, Report
def svm_NL(X_train, y_train, X_test, y_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel='rbf', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


# Navie method is used for Navie model creation and evaluate confusion matrix and send accuracy, Report
def Navie(X_train, y_train, X_test, y_test):
    from sklearn.naive_bayes import GaussianNB
    classifier = GaussianNB()
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


# KNN method is used for KNN model creation and evaluate confusion matrix and send accuracy, Report
def knn(X_train, y_train, X_test, y_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


# Decision method is used for Decision model creation and evaluate confusion matrix and send accuracy, Report
def Decision(X_train, y_train, X_test, y_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


# random method is used for random forest model creation and evaluate confusion matrix and send accuracy, Report
def random(X_train, y_train, X_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
    classifier.fit(X_train, y_train)
    return cm_prediction(classifier, X_test, y_test)


# pca_Classification method is used for create dataset with columns name as
# 'Logistic','SVMl','SVMnl','KNN','Navie','Decision','Random' and index PCA
# and fill the each columns values
def pca_Classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf):

    dataframe = pd.DataFrame(index=['PCA'], columns=['Logistic', 'SVMl', 'SVMnl', 'KNN', 'Navie', 'Decision', 'Random'])
    for number, idex in enumerate(dataframe.index):
        dataframe['Logistic'][idex] = acclog[number]
        dataframe['SVMl'][idex] = accsvml[number]
        dataframe['SVMnl'][idex] = accsvmnl[number]
        dataframe['KNN'][idex] = accknn[number]
        dataframe['Navie'][idex] = accnav[number]
        dataframe['Decision'][idex] = accdes[number]
        dataframe['Random'][idex] = accrf[number]
    return dataframe


# Read data from file and dataset should without index
dataset = pd.read_csv("prep.csv", index_col=None)
df2 = dataset
# Preprocessed by one hot encoding
df2 = pd.get_dummies(df2, drop_first=True)

# assign the input only
indep_X = df2.drop('classification_yes', axis=1)
# assign output only
dep_Y = df2['classification_yes']

# choose the feature selection here using PCA with n_components
pca_features = lda_selection(indep_X, dep_Y, 7)

# Create 7 empty lists for each algorithm and split the input and output
# Evaluate each algorithm's accuracy and send to pca_Classification function
# finally the evaluation data is represented by table view.
acclog = []
accsvml = []
accsvmnl = []
accknn = []
accnav = []
accdes = []
accrf = []


X_train, X_test, y_train, y_test = split_scalar(pca_features, dep_Y)


classifier, Accuracy, report, X_test, y_test, cm = logistic(X_train, y_train, X_test, y_test)
acclog.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = svm_linear(X_train, y_train, X_test, y_test)
accsvml.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = svm_NL(X_train, y_train, X_test, y_test)
accsvmnl.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = knn(X_train, y_train, X_test, y_test)
accknn.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = Navie(X_train, y_train, X_test, y_test)
accnav.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = Decision(X_train, y_train, X_test, y_test)
accdes.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = random(X_train, y_train, X_test, y_test)
accrf.append(Accuracy)

result = pca_Classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf)
print(result)

Explained variance ratio: [1.]
Total variance captured: 1.0
    Logistic  SVMl SVMnl   KNN Navie Decision Random
PCA     0.98  0.98  0.98  0.98  0.98     0.99   0.99


In [6]:
result
#7

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
PCA,0.98,0.98,0.98,0.98,0.98,0.99,0.99
